In [1]:
# %pip install transformers==4.40.1 peft==0.4.0
# %pip install sentencepiece
# %pip install accelerate
# %pip install torch
# %pip install peft
# %pip install datasets
# %pip install bitsandbytes


# 前置

In [1]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel  # 0.5.0
import torch

# Load Models
base_model = "meta-llama/Meta-Llama-3-8B" 
peft_model = "FinGPT/fingpt-mt_llama3-8b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0")
model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cpu")

model = PeftModel.from_pretrained(model, peft_model)
model = model.eval()


/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "
/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizerFast'. 
The class this function is called from is 'LlamaTokenizerFast'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Loading checkpoint shards: 100%|██████████| 4/4 [01:38<00:00, 24.53s/it]


In [2]:
tokenizer.pad_token = tokenizer.eos_token
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def fingpt_gen(prompt):
    print("Start running...")
    # Set device
    tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512).to(device)
    res = model.generate(**tokens, max_length=512)
    res_sentences = [tokenizer.decode(i) for i in res]
    out_text = [o.split("Answer: ")[1] for o in res_sentences]

    # Show results
    for idx, sentiment in enumerate(out_text):
        print(f"Prompt: {prompt[idx]}", end=" ")
        print(f"{sentiment}")

# 問答

In [25]:
# %%time
import json
import os

env = {
    "stock_id" : "NVDA",
    "stock_name" : "Nvidia",
    "country" : "us",
    "start_date" : "20230401",
    "end_date" : "20240331",
}

path = f"{os.path.dirname(os.path.abspath(os.getcwd()))}/history_data/{env['country']}/news_title/{env['stock_id']}news_title.json"
with open(path, "r") as f:
    data = json.load(f)
    # print(data.keys())

news_list = []
for date, news_items in data.items():
    for news in news_items:
        news_list.append(news['headline'])
# print(len(news_list))

news_list = news_list[:10]

prompt = []
for i, news in enumerate(news_list):
    p_text = f"Instruction: What is the sentiment of this news? Please choose an answer from {{negative/neutral/positive}}\nInput: {news}\nAnswer: "
    prompt.append(p_text)
print(prompt)

fingpt_gen(prompt)

/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2692: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


['Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}\nInput: GeForce RTX 3070 Availability Update\nAnswer: ', 'Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}\nInput: NVIDIA GeForce RTX 3070 Ampere Mobility GPU With 8 GB Memory Pictured, RTX 2080 Ti Tier Performance Coming To Laptops Soon!\nAnswer: ', 'Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}\nInput: Get Trained, Go Deep: How Organizations Can Transform Their Workforce into an AI Powerhouse\nAnswer: ', 'Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}\nInput: NVIDIA GeForce RTX 3070 Mobile (GA104) GPU pictured\nAnswer: ', 'Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}\nInput: Alleged NVIDIA GeForce RTX 3070 Mobile Ampere GPU Leaks Into The Wild\n

In [ ]:
# Make prompts
prompt = [
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: 川普關稅政策帶來市場亂流，此外，密西根大學調查顯示，2月消費者預期未來1年通膨為4.3%，創2023年11月以來新高
Answer: ''',
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: Apple Seeks to End Savings & Credit Card Partnership With Goldman Sachs — What Will Happen to Your Accounts?
Answer: '''
]

# Set device
tokenizer.pad_token = tokenizer.eos_token


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512).to(device)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]





# Show results
for sentiment in out_text:
    print(sentiment)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|end_of_text|><|end_of_text|>
Neutral<|end_of_text|>


In [7]:
# %%time
import json
import os

env = {
    "stock_id" : "AAPL",
    "stock_name" : "Apple",
    "country" : "us",
    "start_date" : "20230401",
    "end_date" : "20240331",
}

path = f"{os.path.dirname(os.path.abspath(os.getcwd()))}/history_data/{env['country']}/news_title/{env['stock_id']}news_title.json"
with open(path, "r") as f:
    data = json.load(f)
    # print(data.keys())

news_list = []
for date, news_items in data.items():
    for news in news_items:
        news_list.append(news['headline'])
# print(len(news_list))

news_list = news_list[:10]

prompt = []
for i, news in enumerate(news_list):
    # print(news)
    p_text = f"Instruction: What is the sentiment of this news? Please choose an answer from {{negative/neutral/positive}}\nInput: {news}\nAnswer: "
    prompt.append(p_text)
# print(prompt)

fingpt_gen(prompt)

Start running...


/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2692: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: Founded by Google AI and Apple Siri Engineers, Pixie Labs
Answer:  Neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: Apple TV Channels adds AMC+ streaming to its list of services
Answer:  Neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: AMC Plus Now Available Via Apple, Prime Video
Answer:  neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: Here’s where Biden and Trump stand on antitrust, social media and other tech issues
Answer:  neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: 5 Local Spots to 

In [3]:
# %%time
import json
import os

env = {
    "stock_id" : "AAPL",
    "stock_name" : "Apple",
    "country" : "us",
    "start_date" : "20230401",
    "end_date" : "20240331",
}

path = f"{os.path.dirname(os.path.abspath(os.getcwd()))}/history_data/{env['country']}/news_title/{env['stock_id']}news_title.json"
with open(path, "r") as f:
    data = json.load(f)
    # print(data.keys())

news_list = []
for date, news_items in data.items():
    for news in news_items:
        news_list.append(news['headline'])
# print(len(news_list))

news_list = news_list[:10]

prompt = []
for i, news in enumerate(news_list):
    # print(news)
    p_text = f"Instruction: What is the sentiment of this news? Please choose an answer from {{negative/positive}}\nInput: {news}\nAnswer: "
    prompt.append(p_text)
# print(prompt)

fingpt_gen(prompt)

Start running...


/Users/yanyifu/Documents/_Coding/LLM Predict Stock/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2692: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/positive}
Input: Rumors are swirling online about the upcoming Apple iPhone 12 models
Answer:  Neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/positive}
Input: Lakefront Apple Valley property sells for $935K
Answer:  Neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/positive}
Input: Hurricane advisory in ENC; StormTrack 12 says timing still uncertain
Answer:  Neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/positive}
Input: Why I Still Use an Old PowerPC Mac in 2020
Answer:  Neutral<|end_of_text|>
Prompt: Instruction: What is the sentiment of this news? Please choose an answer from {negative/positive}
Input: Apple surpasses Saudi Aramco as world’s most valuable company
Answer:  Positiv